In [1]:
"""
RS-232 interface to an ASI tiger controller.

Hazen 05/18
"""
import traceback
import serial
import time


class RS232(object):
    """
    The basic RS-232 communication object which is used by all the objects
    that communicate with their associated hardware using RS-232.
    """

    def __init__(self,
                 baudrate = None,
                 encoding = 'utf-8',
                 end_of_line = "\r",
                 port = None,
                 timeout = 1.0e-3,
                 wait_time = 1.0e-2,
                 **kwds):
        """
        port - The port for RS-232 communication, e.g. "COM4".
        timeout - The RS-232 time out value.
        baudrate - The RS-232 communication speed, e.g. 9800.
        end_of_line - What character(s) are used to indicate the end of a line.
        wait_time - How long to wait between polling events before it is decided 
                    that there is no new data available on the port. 
        """
        super().__init__(**kwds)
        self.encoding = encoding
        self.end_of_line = end_of_line
        self.live = True
        self.wait_time = wait_time
        try:
            self.tty = serial.Serial(port, baudrate, timeout = timeout)
            self.tty.flush()
            time.sleep(self.wait_time)
        except serial.serialutil.SerialException as e:
            print("RS232 Error:", type(e), str(e))
            self.live = False

    def commWithResp(self, command):
        """
        Send a command and wait (a little) for a response.
        """
        self.sendCommand(command)
        time.sleep(10 * self.wait_time)
        response = ""
        response_len = self.tty.inWaiting()
        while response_len:
            response += self.read(response_len)
            time.sleep(self.wait_time)
            response_len = self.tty.inWaiting()
        if len(response) > 0:
            return response

    def getResponse(self):
        """
        Wait (a little) for a response.
        """
        response = ""
        response_len = self.tty.inWaiting()
        while response_len:
            response += self.read(response_len)
            time.sleep(self.wait_time)
            response_len = self.tty.inWaiting()
        if len(response) > 0:
            return response

    def getStatus(self):
        """
        Return True/False if the port open and can we talk to the hardware.
        """
        return self.live

    def read(self, response_len):
        response = self.tty.read(response_len)
        return response.decode(self.encoding)

    def readline(self):
        response = self.tty.readline()
        return response.decode(self.encoding).strip()
        
    def sendCommand(self, command):
        self.tty.flush()
        self.write(command + self.end_of_line)

    def shutDown(self):
        """
        Closes the RS-232 port.
        """
        if self.live:
            self.tty = None

    def waitResponse(self, end_of_response = False, max_attempts = 200):
        """
        Waits much longer for a response. This is the method to use if
        you are sure that the hardware will respond eventually. If you
        don't set end_of_response then it will automatically be the
        end_of_line character, and this will return once it finds the
        first end_of_line character.
        """
        if not end_of_response:
            end_of_response = str(self.end_of_line)
        attempts = 0
        response = ""
        index = -1
        while (index == -1) and (attempts < max_attempts):
            response_len = self.tty.inWaiting()
            if response_len > 0:
                response += self.read(response_len)
            time.sleep(self.wait_time)
            index = response.find(end_of_response)
            attempts += 1
        return response

    def write(self, string):
        self.tty.write(string.encode(self.encoding))

    def writeline(self, string):
        msg = string + self.end_of_line
        self.tty.write(msg.encode(self.encoding))


class Tiger(RS232):
    """
    Tiger controller interface class.
    """
    def __init__(self, **kwds):
        """
        Connect to the tiger controller at the specified port.
        """
        self.live = True
        self.unit_to_um = 0.1
        self.um_to_unit = 1.0/self.unit_to_um

        # FIXME: Why are we storing the position?
        self.x = 0
        self.y = 0
        self.z = 0

        # Try and connect to the controller.
        try:
            super().__init__(**kwds)
            assert not (self.commWithResp("WHO") == None)
            self.joystickOnOff(True)#BB edit

        except (AttributeError, AssertionError):
            print(traceback.format_exc())
            self.live = False
            print("Tiger controller is not connected? Controller is not on?")
            print("Failed to connect to the tiger controller at port", kwds["port"])

    def goAbsolute(self, x, y):
        self.commWithResp("M X={0:.1f} Y={1:.1f}".format(x * self.um_to_unit, y * self.um_to_unit))

    def goRelative(self, x, y):
        self.commWithResp("R X={0:.1f} Y={1:.1f}".format(x * self.um_to_unit, y * self.um_to_unit))

    def jog(self, x_speed, y_speed):
        # Stage velocities are in units of mm/sec.
        vx = x_speed * self.um_to_unit * 1.0e-3
        vy = y_speed * self.um_to_unit * 1.0e-3
        self.commWithResp("VE X={0:.3f} Y={1:.3f}".format(vx, vy))

    def joystickOnOff(self, on):
        # This also turns off the stage motors to disable position
        # feedback control during movies.
        if on:
            try:
                self.commWithResp("J X+ Y+")
                self.commWithResp("MC X+ Y+ Z+")
            except:
                print("Failed communication...")
        else:
            try:
                self.commWithResp("J X- Y-")
                self.commWithResp("MC X- Y- Z-")
            except:
                print("Failed communication...")

    def position(self):
        try:
            [self.x, self.y] = map(lambda x: float(x)*self.unit_to_um, 
                                   self.commWithResp("W X Y").split(" ")[1:3])
        except:
            [self.x, self.y] = [0,0]#BBedit
        return {"x" : self.x,
                "y" : self.y}

    def setLED(self, address, channel, power):
        self.commWithResp(address + "LED " + channel + "={0:0d}".format(int(power)))

    def setTTLMode(self, address, mode):
        self.commWithResp(address + "TTL X={0:0d}".format(int(mode)))

    def setVelocity(self, x_vel, y_vel):
        """
        Set the maximum X/Y speed in mm/sec.
        """
        self.commWithResp("S X={0:.2f} Y={1:.2f}".format(x_vel, y_vel))

    def zero(self):
        self.commWithResp("H X Y")

    def zConfigurePiezo(self, axis, mode):
        self.commWithResp("PM " + str(axis) + "=" + str(mode))
        
    def zMoveTo(self, z):
        """
        Move the z stage to the specified position (in microns).
        """
        self.commWithResp("M Z={0:.2f}".format(z * self.um_to_unit))

    def zPosition(self):
        """
        Query for current z position in microns.
        """
        new_z = self.z
        try:
            temp = self.commWithResp("W Z")
            new_z = float(temp.split(" ")[1])*self.unit_to_um
        except ValueError:
            print("Tiger.zPosition(): could not parse -", temp, "-")            
        self.z = new_z
        return {"z" : self.z}

    def zSetVelocity(self, z_vel):
        """
        Set the maximum Z speed in mm/sec.
        """
        self.commWithResp("S Z={0:.2f}".format(z_vel))
        
    def zZero(self):
        self.commWithResp("H Z")
    def goZRelative(self, z):
        self.commWithResp("R Z={0:.1f}".format(z * self.um_to_unit))

In [2]:

stage = Tiger(port = "COM3", baudrate = 115200)

In [36]:
import zarr
import numpy as np
ims=[]
for c in np.arange(4)+1:
    movie = zarr.open_array(rf"D:\BigBrain\TestBB3\eachcol_0009_c{c}.zarr", mode="r")
    im = np.array(movie,dtype=np.float32)**2
    ims.append(im)
#movie = zarr.open_array(r"C:\Data\movie_0027_c2.zarr", mode="r")
#im2 = np.array(movie,dtype=np.float32)**2
ims= np.array(ims)
import napari
V = napari.Viewer()
V.add_image(ims,contrast_limits=[0,3000])

In [38]:
import zarr
import numpy as np
ims=[]
for c in np.arange(4)+1:
    movie = zarr.open_array(rf"D:\BigBrain\TestBB3\eachcol_0009_c{c}.zarr", mode="r")
    im = np.array(movie,dtype=np.float32)**2
    ims.append(im)
#movie = zarr.open_array(r"C:\Data\movie_0027_c2.zarr", mode="r")
#im2 = np.array(movie,dtype=np.float32)**2
ims1= np.array(ims)

import zarr
import numpy as np
ims=[]
for c in np.arange(4)+1:
    movie = zarr.open_array(rf"D:\BigBrain\TestBB3\eachcol_0010_c{c}.zarr", mode="r")
    im = np.array(movie,dtype=np.float32)**2
    ims.append(im)
#movie = zarr.open_array(r"C:\Data\movie_0027_c2.zarr", mode="r")
#im2 = np.array(movie,dtype=np.float32)**2
ims2= np.array(ims)
import napari
V = napari.Viewer()
V.add_image(ims1,contrast_limits=[0,6000])
V.add_image(ims2,contrast_limits=[0,6000])

C:\ProgramData\anaconda3\Lib\site-packages\napari\_qt\qt_event_loop.py:49: UserWarning: System theme detection requires a Qt6 backend. Please switch to PyQt6 or PySide6 to use it.
  theme_type=get_system_theme(),


<Image layer 'ims2' at 0x1d631402c10>

In [37]:
import napari
V = napari.Viewer()
V.add_image(ims,contrast_limits=[0,3000])

C:\ProgramData\anaconda3\Lib\site-packages\napari\_qt\qt_event_loop.py:49: UserWarning: System theme detection requires a Qt6 backend. Please switch to PyQt6 or PySide6 to use it.
  theme_type=get_system_theme(),
C:\ProgramData\anaconda3\Lib\site-packages\napari\_qt\qt_event_loop.py:49: UserWarning: System theme detection requires a Qt6 backend. Please switch to PyQt6 or PySide6 to use it.
  theme_type=get_system_theme(),


<Image layer 'ims' at 0x1d62e402e90>

In [33]:
import zarr
import numpy as np
ims=[]
for c in np.arange(4)+1:
    movie = zarr.open_array(rf"D:\BigBrain\TestBB2\eachcol_0001_c{c}.zarr", mode="r")
    im = np.array(movie,dtype=np.float32)**2
    ims.append(im)
#movie = zarr.open_array(r"C:\Data\movie_0027_c2.zarr", mode="r")
#im2 = np.array(movie,dtype=np.float32)**2
ims1= np.array(ims)

import zarr
import numpy as np
ims=[]
for c in np.arange(4)+1:
    movie = zarr.open_array(rf"D:\BigBrain\TestBB2\eachcol_0005_c{c}.zarr", mode="r")
    im = np.array(movie,dtype=np.float32)**2
    ims.append(im)
#movie = zarr.open_array(r"C:\Data\movie_0027_c2.zarr", mode="r")
#im2 = np.array(movie,dtype=np.float32)**2
ims2= np.array(ims)


import zarr
import numpy as np
ims=[]
for c in np.arange(4)+1:
    movie = zarr.open_array(rf"D:\BigBrain\TestBB2\eachcol_0006_c{c}.zarr", mode="r")
    im = np.array(movie,dtype=np.float32)**2
    ims.append(im)
#movie = zarr.open_array(r"C:\Data\movie_0027_c2.zarr", mode="r")
#im2 = np.array(movie,dtype=np.float32)**2
ims3= np.array(ims)


import zarr
import numpy as np
ims=[]
for c in np.arange(4)+1:
    movie = zarr.open_array(rf"D:\BigBrain\TestBB2\eachcol_0007_c{c}.zarr", mode="r")
    im = np.array(movie,dtype=np.float32)**2
    ims.append(im)
#movie = zarr.open_array(r"C:\Data\movie_0027_c2.zarr", mode="r")
#im2 = np.array(movie,dtype=np.float32)**2
ims4= np.array(ims)


import zarr
import numpy as np
ims=[]
for c in np.arange(4)+1:
    movie = zarr.open_array(rf"D:\BigBrain\TestBB2\eachcol_0008_c{c}.zarr", mode="r")
    im = np.array(movie,dtype=np.float32)**2
    ims.append(im)
#movie = zarr.open_array(r"C:\Data\movie_0027_c2.zarr", mode="r")
#im2 = np.array(movie,dtype=np.float32)**2
ims5= np.array(ims)

In [35]:
import napari
V = napari.Viewer()
V.add_image(ims1,contrast_limits=[0,7000])
V.add_image(ims2,contrast_limits=[0,7000])
V.add_image(ims3,contrast_limits=[0,7000])
V.add_image(ims4,contrast_limits=[0,7000])
V.add_image(ims5,contrast_limits=[0,7000])

C:\ProgramData\anaconda3\Lib\site-packages\napari\_qt\qt_event_loop.py:49: UserWarning: System theme detection requires a Qt6 backend. Please switch to PyQt6 or PySide6 to use it.
  theme_type=get_system_theme(),


<Image layer 'ims5' at 0x1d62045bed0>

In [1]:
#pip install napari[all]

In [3]:

stage.commWithResp("J *?")

':A X=2 Y=3 Z=22 O=23 P=0 E=0 \r\n'

In [3]:

stage.commWithResp("J *?")

':A X=2 Y=3 Z=22 O=0 P=0 E=0 \r\n'

In [4]:

stage.commWithResp("J O=23")

':A \r\n'

In [5]:

stage.commWithResp("HERE O=1")

':A\r\n'

In [ ]:
stage.commWithResp("HERE O=1")

In [6]:
stage.commWithResp("H O+")

':A\r\n'

In [ ]:
print(cmd("BU X"))

In [9]:
txt = stage.commWithResp("S X? Y?")
vx=float(txt.split('X=')[-1].split(' ')[0])
vy=float(txt.split('Y=')[-1].split(' ')[0])
vx,vy

(1.920015, 1.920015)

In [4]:
print(stage.commWithResp("S X? Y?"))

:A X=1.920015 Y=1.920015 



In [3]:
txt = stage.commWithResp("S X? Y?")
float(txt.split('X=')[-1].split(' ')[0])
float(txt.split('Y=')[-1].split(' ')[0])

1.920015

In [218]:
(3200-2600)/2

300.0

In [219]:
3200-300

2900

In [11]:
txt = stage.commWithResp("BU X")
print(txt.replace('\r','\n'))

TIGER_COMM
Motor Axes: X Y Z O P E
Axis Types: x x z o p g
Axis Addr: 1 1 2 2 4 6
Hex Addr: 31 31 32 32 34 36
Axis Props:   6   6   4   4  34   0




In [12]:
txt = stage.commWithResp("2BU X")
print(txt.replace('\r','\n'))

STDZ_OBJ_SLDR_ADJB_ENC2
Motor Axes: Z O 
Axis Types: z o 
Axis Addr: 2 2 
Hex Addr: 32 32
Axis Props:   4   4
CMDS: ZO
BootLdr V:NONE
Hdwr REV.F
POSITIONS SAVED
CLOCKED POSITIONS
ENC_INT_2
SAFE_TURRET
SCAN MODULE
SEEK_LIMITS
SHUTDOWN_TASK
CLUTCH XYKNOB FASTSLOW
ZF_KNOB





In [13]:
txt = stage.commWithResp("2CCA X?")
print(txt.replace('\r','\n'))

:A  Z:L1Ao O:Fo
9  Z 1 SCOPE 100u/T
10 Z 2 SCOPE 200u/T
19 Z 4 SCOPE 400u/T
11 Z B PITCH 4/in
12 Z A PITCH 16/in
13 Z C PITCH 2/in
14 Z U PITCH 40/in
47 Z D PITCH 1/in
29 Z S Siskiyou Motor
26 Z 1 ZLE 10nm
27 Z 2 ZLE 20nm
32 Z o LIMITS NO
33 Z c LIMITS NC
60 Z D 500 C/rev
61 Z M 1000 C/rev
62 Z 2 2048 C/rev





In [14]:
txt = stage.commWithResp("UM Z?")
print(txt.replace('\r','\n'))

:Z=10000.000000 A




In [16]:
txt = stage.commWithResp("W Z")
print(txt.replace('\r','\n'))

:A 2058 




In [17]:
txt = stage.commWithResp("2BU X")
print(txt.replace('\r','\n'))

STDZ_OBJ_SLDR_ADJB_ENC2
Motor Axes: Z O 
Axis Types: z o 
Axis Addr: 2 2 
Hex Addr: 32 32
Axis Props:   4   4
CMDS: ZO
BootLdr V:NONE
Hdwr REV.F
POSITIONS SAVED
CLOCKED POSITIONS
ENC_INT_2
SAFE_TURRET
SCAN MODULE
SEEK_LIMITS
SHUTDOWN_TASK
CLUTCH XYKNOB FASTSLOW
ZF_KNOB





In [18]:
txt = stage.commWithResp("2TTL X?")
print(txt.replace('\r','\n'))

:A X=100 




In [19]:
cmd = stage.commWithResp

In [21]:
float(cmd("UM Z?").split("=")[1].split()[0])

10000.0

In [35]:
### 
# Read current position and preserve the existing speed.
um_z = float(cmd("UM Z?").split("=")[1].split()[0])
z0_mm = float(cmd("W Z").split()[1]) / um_z
old_speed = float(cmd("S Z?").split("=")[1].split()[0])

In [36]:
old_speed

0.01

In [34]:
print(cmd("S Z=0.01"))

:A 



In [40]:
old_speed,z0_mm,um_z

(0.01, 1.0604, 10000.0)

In [39]:
float(cmd("W Z").split()[1]) / um_z

1.0604

In [25]:
# Z is axis index 0 on card 2.
# Slow-axis index 9 selects a Z-only scan.
print(cmd("2SCAN Y=0 Z=9 F=0"))

# One scan; disable additional backlash overshoot.
print(cmd("2SCANV X=0 Y=0 Z=1 F=10 T=0"))

:A 

:A 



In [37]:
# Scan 10 µm; output every 100 × 10 nm = 1 µm.
print(cmd(
    f"2SCANR X={z0_mm:.6f} Y={z0_mm + 0.010:.6f} Z=100"
))

:A 



In [41]:
f"2SCANR X={z0_mm:.6f} Y={z0_mm + 0.010:.6f} Z=100"

'2SCANR X=1.060400 Y=1.070400 Z=100'

In [28]:
f"2SCANR X={z0_mm:.6f} Y={z0_mm + 0.010:.6f} Z=100"

'2SCANR X=6.893500 Y=6.903500 Z=100'

In [42]:
print(cmd("2SCAN"))

:A 



In [ ]:
### Clean code

In [ ]:
##### Initial Configuration. I want to only do this once I start hal and load the Tiger stage
# Z is axis index 0 on card 2.
# Slow-axis index 9 selects a Z-only scan.
cmd = stage.commWithResp
print(cmd("2SCAN Y=0 Z=9 F=0"))
# One scan; disable additional backlash overshoot.
print(cmd("2SCANV X=0 Y=0 Z=1 F=10 T=0"))
um_z = float(cmd("UM Z?").split("=")[1].split()[0])

In [168]:
###Based on the following settings which I want to get from the paramter files I want to configure the speed. I want to configure this everythime I load another parmater file into hall
import numpy as np
step_size_um=0.5
travel_distance_um = 20
fps = 40

speed_mm_per_s = step_size_um*fps/1000

cmd(f"S Z={speed_mm_per_s:.6f}") # 0.01 mm/s = 10 µm/s.

':A \r\n'

In [169]:
speed_mm_per_s

0.02

In [213]:
#### I want to decide 

### current position in mm
z0_mm = float(cmd("W Z").split()[1]) / um_z
Zpulse = int(np.round(step_size_um/0.01))#10nm step
if Zpulse==0: Zpulse=1
offset = 3./1000 ### determined empyrically
scan_setup_cmd = f"2SCANR X={z0_mm+offset:.6f} Y={z0_mm+offset + travel_distance_um/1000.:.6f} Z={Zpulse}"
print(scan_setup_cmd)
print(cmd(scan_setup_cmd))
print(cmd("2SCAN"))

2SCANR X=0.018200 Y=0.038200 Z=50
:A 



In [203]:
print(cmd("2SCANR R?"))

:A R=67.000000 



In [228]:
print(cmd("2SCAN"))

:A 



In [215]:
zf_mm = float(cmd("W Z").split()[1]) / um_z
(z0_mm-zf_mm)*1000

0.0

In [46]:
0.5/0.01

50.0

In [59]:
step_size_um=0.5
fps = 40
speed_mm_per_s = step_size_um*fps/1000
speed_mm_per_s 

0.02

In [32]:
import glob,os,numpy as np
fls = glob.glob(r'D:\BigBrain\TestBB\eachcol_0002_c2.zarr\c\*\*\*')
np.sum(np.diff(np.sort([os.path.getctime(fl) for fl in fls])))

np.float64(1.1379954814910889)

In [240]:
fls

['C:\\Data\\movie_0010_c1.zarr\\c\\0\\0\\0',
 'C:\\Data\\movie_0010_c1.zarr\\c\\1\\0\\0',
 'C:\\Data\\movie_0010_c1.zarr\\c\\10\\0\\0',
 'C:\\Data\\movie_0010_c1.zarr\\c\\11\\0\\0',
 'C:\\Data\\movie_0010_c1.zarr\\c\\12\\0\\0',
 'C:\\Data\\movie_0010_c1.zarr\\c\\13\\0\\0',
 'C:\\Data\\movie_0010_c1.zarr\\c\\14\\0\\0',
 'C:\\Data\\movie_0010_c1.zarr\\c\\15\\0\\0',
 'C:\\Data\\movie_0010_c1.zarr\\c\\16\\0\\0',
 'C:\\Data\\movie_0010_c1.zarr\\c\\17\\0\\0',
 'C:\\Data\\movie_0010_c1.zarr\\c\\18\\0\\0',
 'C:\\Data\\movie_0010_c1.zarr\\c\\19\\0\\0',
 'C:\\Data\\movie_0010_c1.zarr\\c\\2\\0\\0',
 'C:\\Data\\movie_0010_c1.zarr\\c\\20\\0\\0',
 'C:\\Data\\movie_0010_c1.zarr\\c\\21\\0\\0',
 'C:\\Data\\movie_0010_c1.zarr\\c\\22\\0\\0',
 'C:\\Data\\movie_0010_c1.zarr\\c\\23\\0\\0',
 'C:\\Data\\movie_0010_c1.zarr\\c\\24\\0\\0',
 'C:\\Data\\movie_0010_c1.zarr\\c\\25\\0\\0',
 'C:\\Data\\movie_0010_c1.zarr\\c\\26\\0\\0',
 'C:\\Data\\movie_0010_c1.zarr\\c\\27\\0\\0',
 'C:\\Data\\movie_0010_c1.zarr\\c\\28